# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To inspect the dataset structure, list all record sets present and their available fields and columns by `@id`. 

In [ ]:
# List all record sets and their fields/columns by @id
record_sets = list(dataset.record_sets)
print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}  |  Name: {field.name}")
    if hasattr(rs, 'columns') and rs.columns is not None:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - @id: {col.id}  |  Name: {col.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Tip:** Replace the `example_record_set_id` below with one of the available record set `@id`s from the overview cell above.

In [ ]:
# Extract data from each record set
# (Auto-list all record set IDs for user's convenience)
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from Record Set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

# Show one (the first) non-empty dataframe's columns as an example
example_record_set_id = next(iter(dataframes))
print(f"\nColumns available in record set {example_record_set_id}:")
print(dataframes[example_record_set_id].columns.tolist())

# Display top rows
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Tip:** Replace `numeric_field_id` and `group_field_id` below with appropriate column names (i.e. from the DataFrame columns corresponding to the `@id` of the field you want to analyze).

In [ ]:
# Choose a DataFrame and fields for EDA
record_set_id = example_record_set_id  # Use the example record set loaded above
df = dataframes[record_set_id]
# Display available columns for reference
print(f"Available columns: {df.columns.tolist()}")

# Choose a numeric column by its @id (update as appropriate)
numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns)>0 else df.columns[0]

# Set a threshold for filtering - adjust accordingly
threshold = 0  # Update as appropriate for your data

if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    pprint.pprint(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    text_columns = df.select_dtypes(include='object').columns.tolist()
    group_field_id = text_columns[0] if text_columns else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Tip:** Choose appropriate fields to visualize using their `@id` column name from the DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Scatter plot comparing numeric field to another field, if available
potential_x = [col for col in df.columns if col != numeric_field_id]
if len(potential_x) > 0 and pd.api.types.is_numeric_dtype(df[potential_x[0]]):
    plt.figure(figsize=(8,5))
    sns.scatterplot(data=df, x=potential_x[0], y=numeric_field_id)
    plt.title(f"{numeric_field_id} vs {potential_x[0]}")
    plt.xlabel(potential_x[0])
    plt.ylabel(numeric_field_id)
    plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² dataset, explored its record set structure, examined field `@id` references, and performed EDA & simple visualizations. Continue your analysis by refining field selections and exploring domain-specific trends based on the provided field `@id` structure.